# Flight Delay ML Platform
## Project Design & Engineering Setup

### 1. Project Overview

This project aims to develop an end-to-end machine learning platform for predicting departure and arrival delay risks for U.S. domestic flights.

Using historical flight performance data from the U.S. Bureau of Transportation Statistics (BTS), the platform will integrate data engineering, feature engineering, machine learning, model evaluation, MLOps, backend services, and AI-assisted analytics.

### Project Objectives

1. Build a reproducible ETL pipeline from raw BTS data.
2. Develop leakage-aware features using flight schedules and historical performance data.
3. Train and compare multiple machine learning models.
4. Evaluate models using chronological splits, classification metrics, probability calibration, and error analysis.
5. Track experiments and model versions using MLflow.
6. Deploy prediction services using FastAPI and Docker.
7. Build an interactive dashboard and a tool-calling AI assistant.

### Key Technologies

- **Language:** Python 3.12
- **Data Engineering:** Pandas, DuckDB, PySpark
- **Machine Learning:** Scikit-learn, LightGBM, XGBoost
- **Experiment Tracking:** MLflow
- **Backend:** FastAPI
- **Frontend:** To be determined
- **Deployment:** Docker, GitHub Actions
- **AI Integration:** LLM Tool Calling

The project prioritizes reproducibility, leakage prevention, model reliability, and software engineering practices rather than model complexity alone.

## 2. Prediction Tasks and Business Context

The platform supports two related binary classification tasks.

### Task A: Departure Delay Prediction (Primary)

**Question:**
Will a flight depart from its origin airport gate at least 15 minutes later than scheduled?

- Target column: `DepDel15`
- Positive class: Departure delay >= 15 minutes
- Negative class: Departure delay < 15 minutes

### Task B: Arrival Delay Prediction (Secondary)

**Question:**
Will a flight arrive at its destination airport gate at least 15 minutes later than scheduled?

- Target column: `ArrDel15`
- Positive class: Arrival delay >= 15 minutes
- Negative class: Arrival delay < 15 minutes

### Prediction Timestamp

Both predictions are generated **24 hours before scheduled gate departure**, based on the origin airport's local time.

Only information available at this prediction timestamp may be used.

### Example

Consider a flight from SFO to JFK.

- Scheduled departure: 3:00 PM (Pacific Time)
- Prediction timestamp: 3:00 PM on the previous day (Pacific Time)
- Scheduled arrival: 11:30 PM (Eastern Time)

The platform generates two probabilities:

1. Probability of departure delay >= 15 minutes.
2. Probability of arrival delay >= 15 minutes.

### Business Value

**Departure Delay Risk**
- Helps passengers assess potential disruption before traveling.
- Supports early trip planning.

**Arrival Delay Risk**
- Helps passengers assess connection risks.
- Supports destination transportation and arrival planning.

### Important Limitation

Departure delay is not the same as boarding delay.

BTS reports gate departure and gate arrival times, but does not directly provide scheduled and actual boarding start times.

Therefore, predicted departure delays must not be presented as official boarding time changes.

## 3. Data Scope and Feature Availability

### 3.1 Data Source

**Source:** Bureau of Transportation Statistics (BTS)

**Dataset:** Reporting Carrier On-Time Performance

Official website:
https://www.transtats.bts.gov/

### 3.2 Initial Data Scope

- Period: January 2023 - December 2025
- Flight type: U.S. domestic scheduled flights
- Origin airports: ATL, DFW, DEN, ORD, LAX, SFO
- Destinations: All eligible destinations from these origin airports
- Raw format: CSV / ZIP
- Processed format: Parquet

The initial data audit will use a small subset before scaling to the complete dataset.

Historical warm-up data from 2022 may be added when required for historical feature engineering.

### 3.3 Initial Modeling Population

The initial evaluation population includes flights that:

- Were not cancelled.
- Were not diverted.
- Have valid departure and arrival delay labels.

Cancellation and diversion records will be retained for data quality analysis.

Because this filtering uses final flight outcomes, the initial evaluation applies to the completed-flight population. It does not measure unconditional delay risk across all booked flights.

### 3.4 Baseline Features

The initial schedule-based feature candidates include:

- `FlightDate`
- `Reporting_Airline`
- `Origin`
- `Dest`
- `CRSDepTime`
- `CRSArrTime`
- `CRSElapsedTime`
- `Distance`

Derived calendar features may include:

- Month
- Day of week
- Scheduled departure hour
- Weekend indicator

### 3.5 Data Leakage Prevention

The following current-flight outcome fields must not be used as predictors:

- `DepDelay`, `ArrDelay`
- `DepDel15`, `ArrDel15`
- `DepTime`, `ArrTime`
- `WheelsOff`, `WheelsOn`
- `TaxiOut`, `TaxiIn`
- `ActualElapsedTime`
- Cancellation and diversion outcomes
- Post-flight delay cause fields

Historical aggregate features must respect both:

1. Event time: When the historical flight occurred.
2. Availability time: When the historical information became accessible.

Schedule-based fields also require availability checks because retrospective BTS schedules do not necessarily represent the exact schedule known 24 hours before departure.

### 3.6 Known Dataset Limitations

BTS provides retrospective flight records rather than complete historical real-time schedule snapshots.

The project will document assumptions about schedule availability, publication latency, and the supported prediction population.

A leakage-free feature schema alone does not guarantee fully point-in-time-correct historical evaluation.

## 4. Experimental Design

### 4.1 Chronological Data Split

To simulate prediction on future flights, the project uses chronological splits rather than random train-test splitting.

| Dataset | Period | Purpose |
|---|---|---|
| Training | Jan-Dec 2023 | Model fitting |
| Validation | Jan-Aug 2024 | Model selection and hyperparameter tuning |
| Calibration | Sep-Oct 2024 | Probability calibration |
| Threshold | Nov-Dec 2024 | Classification threshold selection |
| Test | Jan-Dec 2025 | Final out-of-time evaluation |

The test set must remain untouched during model development and selection.

Both prediction tasks use the same chronological boundaries but are evaluated independently.

### 4.2 Candidate Models

| Model | Purpose |
|---|---|
| Dummy Classifier | Minimum baseline |
| Logistic Regression | Interpretable baseline |
| Random Forest | Nonlinear benchmark |
| LightGBM | Main gradient boosting candidate |
| XGBoost | Alternative gradient boosting model |

More complex models will only be introduced if supported by experimental evidence.

### 4.3 Evaluation Metrics

**Primary Metric**
- Average Precision (AP), summarizing the precision-recall curve.

**Secondary Metrics**
- Positive-class F1 Score
- Precision
- Recall
- ROC-AUC

**Probability Quality**
- Brier Score
- Calibration Curve

**Error and Robustness Analysis**
- Performance by airport
- Performance by airline
- Performance by month
- Performance by departure time
- Out-of-time performance degradation

### 4.4 Calibration and Threshold Selection

Calibration and classification threshold optimization use separate time periods.

Each prediction task has its own calibration method and decision threshold.

No calibration parameters, thresholds, or hyperparameters may be selected using the final test set.

### 4.5 Research Questions

**RQ1:** How do linear and tree-based models compare under chronological evaluation?

**RQ2:** Do leakage-aware historical features improve out-of-time prediction performance?

**RQ3:** Can probability calibration and threshold optimization improve operational usefulness?

**RQ4:** How do departure delay and arrival delay prediction differ in performance and error patterns?

In [1]:
from datetime import date, timedelta

from flight_delay.features.schema import (
    TARGETS,
    BASELINE_FEATURES,
    validate_baseline_features,
)

# Project configuration
PROJECT_CONFIG = {
    "project_name": "flight-delay-ml-platform",
    "prediction_horizon_hours": 24,
    "origin_airports": [
        "ATL", "DFW", "DEN",
        "ORD", "LAX", "SFO"
    ],
    "splits": {
        "train": ("2023-01-01", "2023-12-31"),
        "validation": ("2024-01-01", "2024-08-31"),
        "calibration": ("2024-09-01", "2024-10-31"),
        "threshold": ("2024-11-01", "2024-12-31"),
        "test": ("2025-01-01", "2025-12-31"),
    },
    "primary_metric": "average_precision",
}

# Validate both prediction targets.
assert TARGETS == {
    "departure": "DepDel15",
    "arrival": "ArrDel15",
}

assert set(TARGETS.values()).isdisjoint(BASELINE_FEATURES)

# Validate the approved baseline feature schema.
validate_baseline_features(BASELINE_FEATURES)

# Validate chronological splits.
previous_end = None

for split_name, (start, end) in PROJECT_CONFIG["splits"].items():
    start_date = date.fromisoformat(start)
    end_date = date.fromisoformat(end)

    assert start_date <= end_date

    if previous_end is not None:
        assert start_date == previous_end + timedelta(days=1)

    previous_end = end_date
    print(f"{split_name:12s}: {start} to {end}")

print("\nPrediction Targets:")
for task, target in TARGETS.items():
    print(f"  {task}: {target}")

print("\nBaseline feature count:", len(BASELINE_FEATURES))
print("Project configuration validation passed!")

train       : 2023-01-01 to 2023-12-31
validation  : 2024-01-01 to 2024-08-31
calibration : 2024-09-01 to 2024-10-31
threshold   : 2024-11-01 to 2024-12-31
test        : 2025-01-01 to 2025-12-31

Prediction Targets:
  departure: DepDel15
  arrival: ArrDel15

Baseline feature count: 8
Project configuration validation passed!


## 5. Repository Architecture

### 5.1 Architecture Overview

The project follows a modular Python package structure with a `src/` layout.

Repository:
https://github.com/Ashley96yy/flight-delay-ml-platform

### 5.2 Main Directories

| Directory | Responsibility |
|---|---|
| `configs/` | Experiment and pipeline configurations |
| `data/raw/` | Original BTS datasets |
| `data/interim/` | Intermediate cleaned datasets |
| `data/processed/` | Feature-ready datasets |
| `notebooks/` | EDA and exploratory experiments |
| `src/flight_delay/data/` | Data ingestion, validation, preprocessing |
| `src/flight_delay/features/` | Feature engineering and leakage prevention |
| `src/flight_delay/models/` | Model training and inference |
| `src/flight_delay/evaluation/` | Metrics, calibration, error analysis |
| `tests/` | Unit and integration tests |
| `docs/` | Project documentation |

### 5.3 Planned Extensions

The following components will be developed in later phases:

- `src/flight_delay/api/`: FastAPI prediction services
- `src/flight_delay/agents/`: AI tool-calling integration
- `frontend/`: Interactive prediction dashboard
- `.github/workflows/`: CI/CD automation

### 5.4 Design Principles

**Separation of Concerns**

Each module has a clearly defined responsibility.

**Reusable Code**

Production logic is implemented in Python modules rather than duplicated across notebooks.

**Training-Serving Consistency**

Training and inference should use the same approved feature transformation logic.

**Testability**

Data processing, feature engineering, model inference, and API services should support automated testing.

**Reproducibility**

Experiments should record data versions, configurations, model parameters, and evaluation results.

## 6. Development Environment

### 6.1 Local Environment

- Operating System: macOS
- Python Version: 3.12.2
- Virtual Environment: `.venv`
- IDE: VS Code
- Package Manager: pip
- Build System: setuptools
- Project Configuration: `pyproject.toml`

### 6.2 Python Package Installation

The project uses a `src/` layout with editable installation.

Installation command:

```bash
python -m pip install -e ".[dev]"
```

Editable installation allows changes to local Python modules to be reflected without reinstalling the entire package.

### 6.3 Development Tools

**pytest 9.2.0**
- Automated testing
- Parameterized test cases
- Regression testing

**Ruff 0.17.4**
- Static code analysis
- Import organization
- Python style and quality checks

### 6.4 Dependency Management

Dependencies are defined in `pyproject.toml`.

Development dependencies are installed through the optional `dev` group.

Additional libraries such as Pandas, PySpark, LightGBM, XGBoost, MLflow, and FastAPI will be added as their corresponding modules are developed.

### 6.5 Environment Isolation

The `.venv/` directory is excluded from version control through `.gitignore`.

This prevents local Python packages from being committed to GitHub.

In [3]:
import sys
from pathlib import Path
from importlib.metadata import version

import flight_delay

print("Python version:")
print(sys.version.split()[0])

print("\nPython executable:")
print(sys.executable)

print("\nProject package location:")
print(flight_delay.__file__)

print("\nDevelopment tools:")
print("pytest:", version("pytest"))
print("ruff:", version("ruff"))

# Verify expected Python version.
assert sys.version_info[:2] == (3, 12)

# Verify the regular Python package is available.
assert flight_delay.__file__ is not None
assert Path(flight_delay.__file__).name == "__init__.py"

print("\nEnvironment verification passed!")

Python version:
3.12.2

Python executable:
/Users/dingyuyao/Documents/Project/flight-delay-ml-platform/.venv/bin/python

Project package location:
/Users/dingyuyao/Documents/Project/flight-delay-ml-platform/src/flight_delay/__init__.py

Development tools:
pytest: 9.2.0
ruff: 0.17.4

Environment verification passed!


## 7. Unit Testing and Data Leakage Prevention

### 7.1 Objective

Establish an initial automated testing framework to verify feature schema integrity and reduce the risk of target leakage.

### 7.2 Feature Schema Validation

Implementation:

`src/flight_delay/features/schema.py`

The baseline uses an explicit feature allowlist.

Only approved schedule-based fields are permitted.

Unexpected fields are rejected through `validate_baseline_features()`.

### 7.3 Unit Tests

Test file:

`tests/test_feature_schema.py`

The initial tests verify:

**Test 1: Target Exclusion**
- `DepDel15` and `ArrDel15` are not included in baseline features.

**Test 2: Valid Feature Acceptance**
- Approved baseline features pass validation.

**Test 3: Leakage Feature Rejection**
- Current-flight outcome fields are rejected.

Rejected examples:
- `DepDelay`
- `ArrDelay`
- `DepTime`
- `ArrTime`
- `DepDel15`
- `ArrDel15`

The third test uses pytest parameterization to validate multiple forbidden columns.

### 7.4 Test Execution

Command:

```bash
python -m pytest tests/test_feature_schema.py -v
```

Result:

**8 tests passed.**

### 7.5 Code Quality

Command:

```bash
ruff check src/ tests/
```

Result:

**All checks passed.**

### 7.6 Limitations

These tests validate the initial feature schema only.

They do not yet verify:

- Historical feature availability
- Point-in-time correctness
- Data publication delays
- Training-serving consistency
- End-to-end leakage prevention

These checks will be added during data ingestion and feature engineering.

## 8. Development Roadmap

### Phase 0: Project Design

Status: Completed

- Defined departure and arrival delay targets
- Selected prediction timestamp
- Defined initial BTS data scope
- Established chronological evaluation strategy
- Documented initial data leakage policy

### Phase 1: Repository & Environment

Status: In Progress

Completed:
- Repository structure
- Python virtual environment
- Editable package installation
- pytest and Ruff configuration
- Initial feature schema validation tests

Remaining:
- First Git commit and GitHub push

### Phase 2: BTS Data Engineering

Planned:
- Download pilot BTS dataset
- Inspect source schema and data types
- Audit missing values and duplicates
- Validate target labels and flight outcomes
- Build reproducible ETL pipeline
- Export processed Parquet data

### Phase 3: Departure Delay Modeling

Planned:
- Dummy baseline
- Logistic Regression
- Random Forest
- LightGBM / XGBoost
- Chronological model evaluation

### Phase 4: Advanced Feature Engineering

Planned:
- Historical airport delay statistics
- Historical airline and route features
- Point-in-time validation
- Feature importance and ablation experiments

### Phase 5: Arrival Delay Modeling

Planned:
- Reuse shared preprocessing and features
- Train independent arrival delay models
- Compare prediction performance across tasks

### Phase 6: MLOps & Evaluation

Planned:
- MLflow experiment tracking
- Model versioning
- Probability calibration
- Temporal backtesting
- Error analysis and data drift checks

### Phase 7: Backend & Deployment

Planned:
- FastAPI inference service
- Automated API testing
- Interactive dashboard
- Docker containerization
- GitHub Actions CI/CD

### Phase 8: AI Assistant

Planned:
- LLM tool-calling integration
- Flight risk prediction tools
- Historical route analysis
- Model explanation tools
- Tool validation and error handling

### Phase 9: Portfolio Finalization

Planned:
- Reproducible experiments
- Final model evaluation report
- System architecture diagram
- Complete README
- GitHub portfolio and resume bullet points